### Introducción

Este notebook tendrá como finalidad presentar la ingeniería de características (feature engineering)

### Importación de librerías

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

### Definición de constantes

In [2]:
PATH_DIRECTORIO_DATOS_PREPROCESSED = "../../data/processed"
PATH_DATASET_PREPROCCESSED = f"{PATH_DIRECTORIO_DATOS_PREPROCESSED}/dataset_practica_final_preprocessed.csv"

### Lectura de datos

In [3]:
df_hotel = pd.read_csv(PATH_DATASET_PREPROCCESSED)

### Ingeniería de características y lógica de negocio

Mostramos las primeras y las últimas filas del dataframe, la dimension y los tipos de datos.

In [4]:
# Primeras filas del dataframe

df_hotel.head()

,hotel,is_canceled,lead_time,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,Resort Hotel,0,342,July,27,1,0,0,2,0,...,0,C,C,3,No Deposit,0,Transient,0.0,0,0
1,Resort Hotel,0,737,July,27,1,0,0,2,0,...,0,C,C,4,No Deposit,0,Transient,0.0,0,0
2,Resort Hotel,0,7,July,27,1,0,1,1,0,...,0,A,C,0,No Deposit,0,Transient,75.0,0,0
3,Resort Hotel,0,13,July,27,1,0,1,1,0,...,0,A,A,0,No Deposit,0,Transient,75.0,0,0
4,Resort Hotel,0,14,July,27,1,0,2,2,0,...,0,A,A,0,No Deposit,0,Transient,98.0,0,1


In [5]:
# Últimas filas del dataframe

df_hotel.tail()

,hotel,is_canceled,lead_time,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
118892,City Hotel,0,23,August,35,30,2,5,2,0,...,0,A,A,0,No Deposit,0,Transient,96.14,0,0
118893,City Hotel,0,102,August,35,31,2,5,3,0,...,0,E,E,0,No Deposit,0,Transient,225.43,0,2
118894,City Hotel,0,34,August,35,31,2,5,2,0,...,0,D,D,0,No Deposit,0,Transient,157.71,0,4
118895,City Hotel,0,109,August,35,31,2,5,2,0,...,0,A,A,0,No Deposit,0,Transient,104.40,0,0
118896,City Hotel,0,205,August,35,29,2,7,2,0,...,0,A,A,0,No Deposit,0,Transient,151.20,0,2


In [6]:
# Dimension del dataframe

df_hotel.shape

(118897, 27)

In [7]:
# Información del tipo de datos del dataframe

df_hotel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118897 entries, 0 to 118896
Data columns (total 27 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           118897 non-null  object 
 1   is_canceled                     118897 non-null  int64  
 2   lead_time                       118897 non-null  int64  
 3   arrival_date_month              118897 non-null  object 
 4   arrival_date_week_number        118897 non-null  int64  
 5   arrival_date_day_of_month       118897 non-null  int64  
 6   stays_in_weekend_nights         118897 non-null  int64  
 7   stays_in_week_nights            118897 non-null  int64  
 8   adults                          118897 non-null  int64  
 9   children                        118897 non-null  int64  
 10  babies                          118897 non-null  int64  
 11  meal                            118897 non-null  object 
 12  country         

Para poder realizar el entrenamiento de los modelos tenemos que hacer ajustes en algunas columnas que actualmente son de tipo numérico pero que en realidad representan categorías o se pueden expresar mejor como categorías

In [8]:
# Columna adults

# Transformamos esta columna numérica en una columna categorica que englobe si una persona viaja sola, en pareja o en grupos.

condicion_adultos =[ 
    df_hotel['adults'] <= 1,
    df_hotel['adults'] == 2,
    df_hotel['adults'] >= 3
]

categorias_adultos = ['1 adulto', '2 adultos', '3 o más adultos']

df_hotel['adults_categories'] = np.select(condicion_adultos,categorias_adultos)

In [11]:
# Columnas children y babies

# Creamos columnas categóricas que representen de forma binaria si hay o no hay niños y bebés respectivamente

df_hotel['has_children'] = (df_hotel['children'] > 0).astype('int64')
df_hotel['has_babies'] = (df_hotel['babies'] > 0).astype('int64')

In [13]:
# Columna arrival_date_day_of_month

# Agrupamos los días del mes en tres categorias
# inicio_mes (del dia 1 al 10)
# mitad_mes (del dia 11 al dia 20)
# fin_mes (del día 21 al día 31)

bins_dias = [0, 10, 20, 31]
labels_dias = ['inicio_mes', 'mitad_mes', 'fin_mes']
df_hotel['month_period'] = pd.cut(df_hotel['arrival_date_day_of_month'], bins=bins_dias, labels=labels_dias)



In [12]:
# Columna arrival_date_week_number

# Agrupamos las semanas dependiendo de si es temporada baja, media o alta

# Alta: Verano (Semanas 23-35) y Navidad (Semanas 51-53)
# Media: Primavera/Otoño (11-22 y 36-44)
# Baja: Resto del año
def asignar_temporada(semana):
    if (23 <= semana <= 35) or (semana >= 51):
        return 'temporada_alta'
    elif (11 <= semana <= 22) or (36 <= semana <= 44):
        return 'temporada_media'
    else:
        return 'temporada_baja'

df_hotel['arrival_season'] = df_hotel['arrival_date_week_number'].apply(asignar_temporada)


In [15]:
# Columna arrival_date_month

# Transformamos los meses en trimestres

meses_q1 = ['January', 'February', 'March']
meses_q2 = ['April', 'May', 'June']
meses_q3 = ['July', 'August', 'September']

def asignar_trimestre(mes):
    if mes in meses_q1:
        return 'Q1'
    elif mes in meses_q2:
        return 'Q2'
    elif mes in meses_q3:
        return 'Q3'
    else:
        return 'Q4'

df_hotel['arrival_quarter'] = df_hotel['arrival_date_month'].apply(asignar_trimestre)


In [22]:
# Columna country

# Para reducir la cardinalidad y hacer más efectiva futuras transformaciones como el one hot encoding, creamos una columna nueva llamada 'country_grouped' que tenga 16 categorías: los 15 países con más presencia y una categoría "Other" que englobe al resto

top_15_countries = df_hotel['country'].value_counts().nlargest(15).index

df_hotel['country_grouped'] = df_hotel['country'].where(df_hotel['country'].isin(top_15_countries), 'Rest_of_the_world')

In [23]:
df_hotel

,hotel,is_canceled,lead_time,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,adr,required_car_parking_spaces,total_of_special_requests,adults_categories,has_children,has_babies,arrival_season,month_period,arrival_quarter,country_grouped
0,Resort Hotel,0,342,July,27,1,0,0,2,0,...,0.00,0,0,2 adultos,0,0,temporada_alta,inicio_mes,Q3,PRT
1,Resort Hotel,0,737,July,27,1,0,0,2,0,...,0.00,0,0,2 adultos,0,0,temporada_alta,inicio_mes,Q3,PRT
2,Resort Hotel,0,7,July,27,1,0,1,1,0,...,75.00,0,0,1 adulto,0,0,temporada_alta,inicio_mes,Q3,GBR
3,Resort Hotel,0,13,July,27,1,0,1,1,0,...,75.00,0,0,1 adulto,0,0,temporada_alta,inicio_mes,Q3,GBR
4,Resort Hotel,0,14,July,27,1,0,2,2,0,...,98.00,0,1,2 adultos,0,0,temporada_alta,inicio_mes,Q3,GBR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118892,City Hotel,0,23,August,35,30,2,5,2,0,...,96.14,0,0,2 adultos,0,0,temporada_alta,fin_mes,Q3,BEL
118893,City Hotel,0,102,August,35,31,2,5,3,0,...,225.43,0,2,3 o más adultos,0,0,temporada_alta,fin_mes,Q3,FRA
118894,City Hotel,0,34,August,35,31,2,5,2,0,...,157.71,0,4,2 adultos,0,0,temporada_alta,fin_mes,Q3,DEU
118895,City Hotel,0,109,August,35,31,2,5,2,0,...,104.40,0,0,2 adultos,0,0,temporada_alta,fin_mes,Q3,GBR
